# Calculate Iron (& Other Nutrients)

In [1]:
# Parameters
SAVE_DFS = True

In [2]:
import ast
import os
import pandas as pd
import requests

from dotenv import load_dotenv
from IPython.display import Image
from pathlib import Path
from rapidfuzz import process, fuzz
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
# Load splits from CSVs
df_train = pd.read_csv('../data/train/df_train.csv')
df_val = pd.read_csv('../data/val/df_val.csv')
df_test = pd.read_csv('../data/test/df_test.csv')

df_train.head(2)

,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name
0,https://file.b18a.io/7832973280900104501_54585...,0.8,0.90,oysters,homemade food,['oysters'],{'oysters': '500g'},"{'fat_g': 5.0, 'protein_g': 20.0, 'calories_kc...",raw,20250710,7832973280900104501_545859_.jpeg
1,https://file.b18a.io/7835136777400102715_70587...,0.7,0.95,grilled steak,restaurant food,"['steak', 'broccoli', 'potato', 'tomato', 'sau...","{'steak': '250g', 'broccoli': '50g', 'potato':...","{'fat_g': 30.0, 'protein_g': 50.0, 'calories_k...",grilling,20250702,7835136777400102715_705873_.jpeg


In [29]:
# Load ingredients data (USDA nutrition)
df_ingredients = pd.read_csv('../data/ingredients_auto.csv')
df_ingredients.head(2)

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid",found,food_id,food_name
0,dough sticks,0.84,16.0,0.0,True,2706231.0,"Fish, stick"
1,fried potatoes,0.81,55.0,7.5,True,2709709.0,"Sweet potato fries, NFS"


In [35]:
# Fill NaNs with 0
df_ingredients = df_ingredients.fillna(0)

In [57]:
# Example ingredient lookup
ingredient = 'dates'
df_ingredients.loc[df_ingredients['ingredients'] == ingredient].reset_index().to_dict(orient='index')[0]

{'index': 493,
 'ingredients': 'dates',
 'Iron, Fe': 1.02,
 'Calcium, Ca': 39.0,
 'Vitamin C, total ascorbic acid': 0.4,
 'found': True,
 'food_id': 2709203.0,
 'food_name': 'Date'}

In [ ]:
# Set output file names
out_val_file = '../data/val/df_val_with_iron.csv'
out_test_file = '../data/test/df_test_with_iron.csv'
out_train_file = '../data/train/df_train_with_iron.csv'


In [74]:
def add_nutrients(df, nutrients=['Iron, Fe', 'Calcium, Ca', 'Vitamin C, total ascorbic acid']):
    # Make sure portions are dicts
    df['portion_size'] = df['portion_size'].apply(ast.literal_eval)
    # Inititalize nutrient values
    for nutrient in nutrients:
        df[nutrient] = 0.0

    for row in tqdm(df.itertuples(), total=len(df)):
        portions = row.portion_size
        # print(portions)
        for portion, size in portions.items():
            # print(portion)
            if size.endswith('kg'):
                size = float(size.replace('kg', '')) * 1000
            elif size.endswith('g'):
                size = float(size.replace('g', ''))
            elif size.endswith('ml'):
                size = float(size.replace('ml', ''))
            else:
                print('Invalid size!', portion, size)
                continue
            # print(size)
            # Look up in ingredients df
            ingredients_dict = df_ingredients.loc[df_ingredients['ingredients'] 
                                                == portion].reset_index().to_dict(orient='index')[0]
            # Calculate and add nutrients per 100g (or ml)
            for nutrient in nutrients:
                # print(nutrient, 'val', ingredients_dict[nutrient])
                # print('calc val', ingredients_dict[nutrient] * (size / 100))
                df.loc[row.Index, nutrient] += (ingredients_dict[nutrient] * (size / 100))
    return df

In [ ]:
# Update val data
file = out_val_file
df_val = add_nutrients(df_val)

if SAVE_DFS:
    # Save split to CSV
    df_val.to_csv(file, index=False)

# Load split from CSVs
df_val = pd.read_csv(file)
df_val

 43%|████▎     | 216/500 [00:00<00:00, 1076.47it/s]

Invalid size! juice 250mL
Invalid size! fried egg 1
Invalid size! coconut  1 medium
Invalid size! eggs 2


100%|██████████| 500/500 [00:00<00:00, 1032.78it/s]

Invalid size! eggs  3 large
Invalid size! bananas  10 medium
Invalid size! eggs 2


,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,https://file.b18a.io/7840587201400100529_43779...,0.8,0.9,dates,raw vegetables and fruits,['dates'],{'dates': '100g'},"{'fat_g': 0.15, 'protein_g': 2.5, 'calories_kc...",raw,20250630,7840587201400100529_437799_.jpg,1.020,39.0,0.40
1,https://file.b18a.io/7841301245100105137_12818...,0.7,0.9,rice porridge,homemade food,"['rice', 'water']","{'rice': '200g', 'water': '500ml'}","{'fat_g': 0.5, 'protein_g': 3, 'calories_kcal'...",boiling,20250702,7841301245100105137_128189_.jpg,3.540,45.0,10.80
2,https://file.b18a.io/7868687722300104071_78539...,0.7,0.9,sushi,restaurant food,"['rice', 'seaweed', 'fish', 'sauce']",{'sushi': '150g'},"{'fat_g': 5.0, 'protein_g': 10.0, 'calories_kc...",raw/assembled,20250708,7868687722300104071_785395_.jpeg,0.315,9.0,1.35
3,https://file.b18a.io/7832619357300109067_30937...,0.8,0.9,hot pot,restaurant food,"['meat', 'vegetables', 'noodles', 'spices']","{'meat': '200g', 'vegetables': '150g', 'noodle...","{'fat_g': 30.0, 'protein_g': 40.0, 'calories_k...",boiling,20250722,7832619357300109067_309372_.jpeg,5.120,215.0,12.95
4,https://file.b18a.io/7857094036300106896_65243...,0.8,0.9,stir-fried noodles with chicken and vegetables,restaurant food,"['noodles', 'chicken', 'bok choy', 'bean sprou...","{'noodles': '200g', 'chicken': '100g', 'vegeta...","{'fat_g': 20.0, 'protein_g': 30.0, 'calories_k...",stir-frying,20250628,7857094036300106896_652432_.jpg,3.800,135.0,12.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,https://file.b18a.io/7838324946500106239_55661...,0.7,1.0,vegetable fried noodles,homemade food,"['noodles', 'cabbage', 'carrots', 'eggs', 'soy...","{'noodles': '300g', 'cabbage': '100g', 'carrot...","{'fat_g': 20.0, 'protein_g': 15.0, 'calories_k...",stir-frying,20250712,7838324946500106239_556617_.jpeg,2.630,112.0,38.70
496,https://file.b18a.io/7839233029900100235_14172...,0.8,0.9,spicy noodle soup,homemade food,"['noodles', 'vegetables', 'spices', 'broth']","{'noodles': '200g', 'vegetables': '150g'}","{'fat_g': 15.0, 'protein_g': 20.0, 'calories_k...",boiling,20250627,7839233029900100235_141725_.png,3.800,135.0,12.75
497,https://file.b18a.io/7932583069300101788_62688...,0.7,1.0,mixed asian plate,restaurant food,"['rice', 'chicken', 'egg', 'noodles', 'various...","{'rice': '200g', 'meat': '150g', 'vegetables':...","{'fat_g': 30.0, 'protein_g': 40.0, 'calories_k...",stir-fried and boiled,20250716,7932583069300101788_626880_.jpeg,9.775,282.0,27.95
498,https://file.b18a.io/7840601407900101976_84166...,0.8,0.9,chocolate cake,restaurant food,"['chocolate', 'cream', 'sugar', 'flour', 'eggs']",{'cake': '150g'},"{'fat_g': 20.0, 'protein_g': 5.0, 'calories_kc...",baked,20250628,7840601407900101976_841669_.png,2.235,16.5,0.00


In [82]:
# Update test data
file = out_test_file
df_test = add_nutrients(df_test)

if SAVE_DFS:
    # Save split to CSV
    df_test.to_csv(file, index=False)

# Load split from CSVs
df_test = pd.read_csv(file)
df_val

 44%|████▍     | 221/500 [00:00<00:00, 1120.11it/s]

Invalid size! egg tart  6 pieces
Invalid size! eggs  20 pieces
Invalid size! water 1L
Invalid size! milk 250mL


100%|██████████| 500/500 [00:00<00:00, 1063.00it/s]


,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,https://file.b18a.io/7840587201400100529_43779...,0.8,0.9,dates,raw vegetables and fruits,['dates'],{'dates': '100g'},"{'fat_g': 0.15, 'protein_g': 2.5, 'calories_kc...",raw,20250630,7840587201400100529_437799_.jpg,1.020,39.0,0.40
1,https://file.b18a.io/7841301245100105137_12818...,0.7,0.9,rice porridge,homemade food,"['rice', 'water']","{'rice': '200g', 'water': '500ml'}","{'fat_g': 0.5, 'protein_g': 3, 'calories_kcal'...",boiling,20250702,7841301245100105137_128189_.jpg,3.540,45.0,10.80
2,https://file.b18a.io/7868687722300104071_78539...,0.7,0.9,sushi,restaurant food,"['rice', 'seaweed', 'fish', 'sauce']",{'sushi': '150g'},"{'fat_g': 5.0, 'protein_g': 10.0, 'calories_kc...",raw/assembled,20250708,7868687722300104071_785395_.jpeg,0.315,9.0,1.35
3,https://file.b18a.io/7832619357300109067_30937...,0.8,0.9,hot pot,restaurant food,"['meat', 'vegetables', 'noodles', 'spices']","{'meat': '200g', 'vegetables': '150g', 'noodle...","{'fat_g': 30.0, 'protein_g': 40.0, 'calories_k...",boiling,20250722,7832619357300109067_309372_.jpeg,5.120,215.0,12.95
4,https://file.b18a.io/7857094036300106896_65243...,0.8,0.9,stir-fried noodles with chicken and vegetables,restaurant food,"['noodles', 'chicken', 'bok choy', 'bean sprou...","{'noodles': '200g', 'chicken': '100g', 'vegeta...","{'fat_g': 20.0, 'protein_g': 30.0, 'calories_k...",stir-frying,20250628,7857094036300106896_652432_.jpg,3.800,135.0,12.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,https://file.b18a.io/7838324946500106239_55661...,0.7,1.0,vegetable fried noodles,homemade food,"['noodles', 'cabbage', 'carrots', 'eggs', 'soy...","{'noodles': '300g', 'cabbage': '100g', 'carrot...","{'fat_g': 20.0, 'protein_g': 15.0, 'calories_k...",stir-frying,20250712,7838324946500106239_556617_.jpeg,2.630,112.0,38.70
496,https://file.b18a.io/7839233029900100235_14172...,0.8,0.9,spicy noodle soup,homemade food,"['noodles', 'vegetables', 'spices', 'broth']","{'noodles': '200g', 'vegetables': '150g'}","{'fat_g': 15.0, 'protein_g': 20.0, 'calories_k...",boiling,20250627,7839233029900100235_141725_.png,3.800,135.0,12.75
497,https://file.b18a.io/7932583069300101788_62688...,0.7,1.0,mixed asian plate,restaurant food,"['rice', 'chicken', 'egg', 'noodles', 'various...","{'rice': '200g', 'meat': '150g', 'vegetables':...","{'fat_g': 30.0, 'protein_g': 40.0, 'calories_k...",stir-fried and boiled,20250716,7932583069300101788_626880_.jpeg,9.775,282.0,27.95
498,https://file.b18a.io/7840601407900101976_84166...,0.8,0.9,chocolate cake,restaurant food,"['chocolate', 'cream', 'sugar', 'flour', 'eggs']",{'cake': '150g'},"{'fat_g': 20.0, 'protein_g': 5.0, 'calories_kc...",baked,20250628,7840601407900101976_841669_.png,2.235,16.5,0.00


In [81]:
# Update train data
file = out_train_file
df_train = add_nutrients(df_train)

if SAVE_DFS:
    # Save split to CSV
    df_train.to_csv(file, index=False)

# Load split from CSVs
df_train = pd.read_csv(file)
df_train

 20%|█▉        | 783/4000 [00:00<00:02, 1099.66it/s]

Invalid size! bananas  4 medium (480g)
Invalid size! eggs 2
Invalid size! tomato juice 300mL


 28%|██▊       | 1118/4000 [00:01<00:02, 1082.95it/s]

Invalid size! egg 2 large
Invalid size! eggs 2
Invalid size! drink 100mL
Invalid size! eggs 3
Invalid size! coconut water  1L


 33%|███▎      | 1337/4000 [00:01<00:02, 1075.86it/s]

...

 91%|█████████ | 3647/4000 [00:03<00:00, 933.40it/s] 

Invalid size! milk 250mL
Invalid size! eggs 3


100%|██████████| 4000/4000 [00:03<00:00, 1061.59it/s]

Invalid size! egg 1


,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,https://file.b18a.io/7832973280900104501_54585...,0.8,0.90,oysters,homemade food,['oysters'],{'oysters': '500g'},"{'fat_g': 5.0, 'protein_g': 20.0, 'calories_kc...",raw,20250710,7832973280900104501_545859_.jpeg,0.90,160.0,0.50
1,https://file.b18a.io/7835136777400102715_70587...,0.7,0.95,grilled steak,restaurant food,"['steak', 'broccoli', 'potato', 'tomato', 'sau...","{'steak': '250g', 'broccoli': '50g', 'potato':...","{'fat_g': 30.0, 'protein_g': 50.0, 'calories_k...",grilling,20250702,7835136777400102715_705873_.jpeg,5.85,102.0,36.35
2,https://file.b18a.io/7839412177600109625_80412...,0.8,0.95,sweet and sour potatoes,homemade food,"['potatoes', 'green onions', 'sauce']","{'potatoes': '300g', 'sauce': '50g'}","{'fat_g': 10.0, 'protein_g': 5.0, 'calories_kc...",stir-fried,20250709,7839412177600109625_804127_.jpg,1.26,29.0,45.80
3,https://file.b18a.io/7836602996300103366_23419...,0.8,1.00,hot pot,restaurant food,"['beef', 'pork', 'vegetables', 'sauce']","{'beef': '300g', 'pork': '200g', 'vegetables':...","{'fat_g': 80.0, 'protein_g': 100.0, 'calories_...",boiling,20250707,7836602996300103366_234194_.jpeg,6.93,193.0,48.25
4,https://file.b18a.io/7833047810700105394_87228...,0.8,0.95,stir-fried noodles,homemade food,"['noodles', 'carrots', 'bean sprouts', 'sauce']","{'noodles': '300g', 'vegetables': '100g'}","{'fat_g': 15.0, 'protein_g': 20.0, 'calories_k...",stir-frying,20250713,7833047810700105394_872282_.jpg,3.70,120.0,8.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,https://file.b18a.io/7838856390000104847_87945...,0.8,0.90,cucumber,raw vegetables and fruits,['cucumber'],{'cucumber': '300g'},"{'fat_g': 0.1, 'protein_g': 0.7, 'calories_kca...",raw,20250627,7838856390000104847_879459_.jpg,0.00,48.0,8.40
3996,https://file.b18a.io/7839128531500102319_18272...,0.7,0.90,braised chicken,homemade food,"['chicken', 'soy sauce', 'garlic', 'ginger']","{'chicken': '300g', 'sauce': '50g'}","{'fat_g': 30.0, 'protein_g': 35.0, 'calories_k...",braising,20250721,7839128531500102319_182726_.jpeg,0.12,5.0,0.50
3997,https://file.b18a.io/7833732885600102602_18453...,0.8,0.90,stir-fried meat and vegetables,homemade food,"['meat', 'vegetables', 'sauce']","{'meat': '200g', 'vegetables': '150g'}","{'fat_g': 20.0, 'protein_g': 15.0, 'calories_k...",stir-frying,20250714,7833732885600102602_184531_.jpg,4.42,197.0,12.95
3998,https://file.b18a.io/7832593162900100943_94033...,0.8,0.90,seafood salad,homemade food,"['shrimp', 'squid', 'lettuce', 'tomato', 'carr...","{'shrimp': '100g', 'vegetables': '250g'}","{'fat_g': 15.0, 'protein_g': 25.0, 'calories_k...",raw/prepared,20250722,7832593162900100943_940335_.jpeg,5.32,237.0,45.15
